# BioSight - GPU training on Kaggle

Re-runs the brain-MRI pipeline (JEPA pre-training, MAE baseline, few-label segmentation, U-Net baseline) on a Kaggle GPU with a larger budget.

1. Add Data: **mateuszbuda/lgg-mri-segmentation**
2. Settings -> Accelerator: **GPU**
3. Run All, then download `/kaggle/working/models/*.pt` into `models/` and `/kaggle/working/experiments/*` into `experiments/`.

In [ ]:
!pip -q install timm==1.0.12
import os, glob
raw = [p for p in glob.glob('/kaggle/input/**/kaggle_3m', recursive=True)][0]
os.environ['LGG_RAW'] = raw
os.environ['BIOSIGHT_ROOT'] = '/kaggle/working'
os.makedirs('/kaggle/working/ml', exist_ok=True)
print(raw)

In [ ]:
%%writefile /kaggle/working/ml/nets.py
"""BioSight networks shared by training scripts and the API.

- TokenEncoder: one transformer that runs on ViT image patches (2-D grid) or k-mer DNA tokens (1-D grid).
- MultiScalePredictor: JEPA predictor whose queries (masked positions) cross-attend to
  context tokens at two scales (fine tokens + pooled coarse tokens).
- ViTSeg / SegDecoder: light decoder on top of the image encoder for tumour masks.
- UNet: supervised baseline.
- SeqClassifier: pooled head on top of the DNA encoder.
"""
import copy
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from timm.layers import PatchEmbed
from timm.models.vision_transformer import Block

IMG_SIZE = 128
PATCH = 8
IMG_GRID = (IMG_SIZE // PATCH, IMG_SIZE // PATCH)

KMER = 4
DNA_WINDOW = 99                      # bases per window -> 96 overlapping 4-mers
DNA_TOKENS = DNA_WINDOW - KMER + 1
DNA_GRID = (1, DNA_TOKENS)
PAD_ID = 4 ** KMER                   # 256
MASK_ID = PAD_ID + 1                 # 257
DNA_VOCAB = PAD_ID + 2


def sincos_1d(n, dim):
    pos = torch.arange(n, dtype=torch.float32)[:, None]
    omega = 1.0 / (10000 ** (torch.arange(dim // 2, dtype=torch.float32) / (dim // 2)))
    out = pos * omega[None]
    return torch.cat([out.sin(), out.cos()], dim=1)


def sincos_grid(grid, dim):
    h, w = grid
    if h == 1:
        return sincos_1d(w, dim)
    ys = sincos_1d(h, dim // 2)[:, None, :].expand(h, w, dim // 2)
    xs = sincos_1d(w, dim // 2)[None, :, :].expand(h, w, dim // 2)
    return torch.cat([ys, xs], dim=-1).reshape(h * w, dim)


class KmerEmbed(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.emb = nn.Embedding(DNA_VOCAB, dim)

    def forward(self, x):
        return self.emb(x)


class TokenEncoder(nn.Module):
    def __init__(self, embed, grid, dim, depth, heads):
        super().__init__()
        self.embed = embed
        self.grid = grid
        self.dim = dim
        self.register_buffer("pos", sincos_grid(grid, dim)[None], persistent=False)
        self.blocks = nn.ModuleList([Block(dim, heads, mlp_ratio=4.0, qkv_bias=True) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)

    def tokens(self, x):
        return self.embed(x) + self.pos

    def forward(self, x, keep_idx=None, pre_embedded=False):
        t = x if pre_embedded else self.tokens(x)
        if keep_idx is not None:
            t = t[:, keep_idx]
        for blk in self.blocks:
            t = blk(t)
        return self.norm(t)


def image_encoder(dim=128, depth=6, heads=4):
    embed = PatchEmbed(img_size=IMG_SIZE, patch_size=PATCH, in_chans=3, embed_dim=dim)
    return TokenEncoder(embed, IMG_GRID, dim, depth, heads)


def dna_encoder(dim=128, depth=4, heads=4):
    return TokenEncoder(KmerEmbed(dim), DNA_GRID, dim, depth, heads)


class CrossAttnLayer(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.nq = nn.LayerNorm(dim)
        self.nkv = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.self_block = Block(dim, heads, mlp_ratio=4.0, qkv_bias=True)

    def forward(self, q, kv):
        kv = self.nkv(kv)
        q = q + self.attn(self.nq(q), kv, kv, need_weights=False)[0]
        return self.self_block(q)


class MultiScalePredictor(nn.Module):
    """Predicts target-encoder features at masked positions from the visible context."""

    def __init__(self, enc_dim, grid, pool, dim=96, depth=3, heads=4):
        super().__init__()
        self.grid, self.pool, self.dim = grid, pool, dim
        self.inp = nn.Linear(enc_dim, dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.scale_emb = nn.Parameter(torch.zeros(2, dim))
        coarse_grid = (math.ceil(grid[0] / pool[0]), math.ceil(grid[1] / pool[1]))
        self.register_buffer("pos", sincos_grid(grid, dim), persistent=False)
        self.register_buffer("coarse_pos", sincos_grid(coarse_grid, dim), persistent=False)
        self.layers = nn.ModuleList([CrossAttnLayer(dim, heads) for _ in range(depth)])
        self.out = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, enc_dim))
        nn.init.normal_(self.mask_token, std=0.02)

    def coarse(self, c, ctx_idx):
        h, w = self.grid
        b = c.shape[0]
        g = c.new_zeros(b, h * w, self.dim)
        m = c.new_zeros(1, h * w, 1)
        g[:, ctx_idx] = c
        m[:, ctx_idx] = 1.0
        g = g.transpose(1, 2).reshape(b, self.dim, h, w)
        m = m.transpose(1, 2).reshape(1, 1, h, w)
        gp = F.avg_pool2d(g, self.pool, ceil_mode=True)
        mp = F.avg_pool2d(m, self.pool, ceil_mode=True)
        pooled = (gp / mp.clamp_min(1e-6)).flatten(2).transpose(1, 2)
        valid = mp.flatten() > 0
        return pooled[:, valid] + self.coarse_pos[valid] + self.scale_emb[1]

    def forward(self, ctx, ctx_idx, tgt_idx):
        c = self.inp(ctx)
        fine = c + self.pos[ctx_idx] + self.scale_emb[0]
        kv = torch.cat([fine, self.coarse(c, ctx_idx)], dim=1)
        q = (self.mask_token + self.pos[tgt_idx][None]).expand(ctx.shape[0], -1, -1)
        for layer in self.layers:
            q = layer(q, kv)
        return self.out(q)


def ema_copy(module):
    tgt = copy.deepcopy(module)
    for p in tgt.parameters():
        p.requires_grad_(False)
    return tgt


@torch.no_grad()
def ema_update(target, online, m):
    for pt, po in zip(target.parameters(), online.parameters()):
        pt.mul_(m).add_(po.detach(), alpha=1 - m)


def conv_bn(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class SegDecoder(nn.Module):
    """Light decoder: 16x16 token map -> 128x128 mask logits (3 x2 upsampling stages)."""

    def __init__(self, dim):
        super().__init__()
        self.proj = conv_bn(dim, 96)
        self.up = nn.ModuleList([conv_bn(96, 64), conv_bn(64, 32), conv_bn(32, 16)])
        self.head = nn.Conv2d(16, 1, 1)

    def forward(self, t, grid):
        b, n, d = t.shape
        x = self.proj(t.transpose(1, 2).reshape(b, d, *grid))
        for layer in self.up:
            x = layer(F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False))
        return self.head(x)


class ViTSeg(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = SegDecoder(encoder.dim)

    def forward(self, x):
        return self.decoder(self.encoder(x), self.encoder.grid)


class UNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        ch = [base, base * 2, base * 4, base * 8]
        self.downs = nn.ModuleList()
        cin = 3
        for c in ch:
            self.downs.append(nn.Sequential(conv_bn(cin, c), conv_bn(c, c)))
            cin = c
        self.mid = nn.Sequential(conv_bn(ch[-1], ch[-1] * 2), conv_bn(ch[-1] * 2, ch[-1] * 2))
        self.ups = nn.ModuleList()
        self.upconvs = nn.ModuleList()
        cin = ch[-1] * 2
        for c in reversed(ch):
            self.upconvs.append(nn.ConvTranspose2d(cin, c, 2, stride=2))
            self.ups.append(nn.Sequential(conv_bn(c * 2, c), conv_bn(c, c)))
            cin = c
        self.head = nn.Conv2d(ch[0], 1, 1)

    def forward(self, x):
        skips = []
        for d in self.downs:
            x = d(x)
            skips.append(x)
            x = F.max_pool2d(x, 2)
        x = self.mid(x)
        for upc, up, s in zip(self.upconvs, self.ups, reversed(skips)):
            x = up(torch.cat([upc(x), s], dim=1))
        return self.head(x)


class SeqClassifier(nn.Module):
    def __init__(self, encoder, n_classes):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(nn.LayerNorm(encoder.dim), nn.Linear(encoder.dim, n_classes))

    def features(self, x):
        t = self.encoder(x)
        valid = (x != PAD_ID).unsqueeze(-1).float()
        return (t * valid).sum(1) / valid.sum(1).clamp_min(1.0)

    def forward(self, x):
        return self.head(self.features(x))


# ---------- masking ----------

def block_mask_2d(grid, gen, n_targets=4, t_scale=(0.15, 0.2), aspect=(0.75, 1.5), c_scale=(0.85, 1.0)):
    """I-JEPA style multi-block masking. Returns (context_idx, target_idx) as LongTensors."""
    h, w = grid
    n = h * w

    def sample_block(scale, ar):
        s = scale[0] + (scale[1] - scale[0]) * torch.rand(1, generator=gen).item()
        a = math.exp(math.log(ar[0]) + (math.log(ar[1]) - math.log(ar[0])) * torch.rand(1, generator=gen).item())
        bh = max(1, min(h, int(round(math.sqrt(s * n * a)))))
        bw = max(1, min(w, int(round(math.sqrt(s * n / a)))))
        top = torch.randint(0, h - bh + 1, (1,), generator=gen).item()
        left = torch.randint(0, w - bw + 1, (1,), generator=gen).item()
        m = torch.zeros(h, w, dtype=torch.bool)
        m[top:top + bh, left:left + bw] = True
        return m.flatten()

    tgt = torch.zeros(n, dtype=torch.bool)
    for _ in range(n_targets):
        tgt |= sample_block(t_scale, aspect)
    ctx = sample_block(c_scale, (1.0, 1.0)) & ~tgt
    if ctx.sum() < 8:
        ctx = ~tgt
    return ctx.nonzero().flatten(), tgt.nonzero().flatten()


def span_mask_1d(n, gen, n_targets=3, t_scale=(0.1, 0.15)):
    tgt = torch.zeros(n, dtype=torch.bool)
    for _ in range(n_targets):
        s = t_scale[0] + (t_scale[1] - t_scale[0]) * torch.rand(1, generator=gen).item()
        length = max(2, int(round(s * n)))
        start = torch.randint(0, n - length + 1, (1,), generator=gen).item()
        tgt[start:start + length] = True
    return (~tgt).nonzero().flatten(), tgt.nonzero().flatten()


# ---------- k-mer tokenisation ----------

_BASE = {"A": 0, "C": 1, "G": 2, "T": 3}


def kmer_tokens(seq, window=DNA_WINDOW):
    """Overlapping 4-mer ids for one window; ambiguous bases / padding -> PAD_ID."""
    seq = seq.upper()[:window]
    ids = []
    for i in range(DNA_TOKENS):
        kmer = seq[i:i + KMER]
        if len(kmer) < KMER or any(ch not in _BASE for ch in kmer):
            ids.append(PAD_ID)
            continue
        v = 0
        for ch in kmer:
            v = v * 4 + _BASE[ch]
        ids.append(v)
    return ids


def seq_windows(seq, stride=50):
    seq = seq.upper()
    if len(seq) <= DNA_WINDOW:
        return [seq]
    starts = list(range(0, len(seq) - DNA_WINDOW + 1, stride))
    if starts[-1] != len(seq) - DNA_WINDOW:
        starts.append(len(seq) - DNA_WINDOW)
    return [seq[s:s + DNA_WINDOW] for s in starts]


In [ ]:
%%writefile /kaggle/working/ml/prepare_data.py
"""Prepare the LGG brain-MRI data.

Reads the Kaggle LGG folders (data/raw/lgg/kaggle_3m/<patient>/*.tif), resizes every slice to 128x128,
splits by patient (seeded 80/20) and writes:
  data/raw/lgg128.npz          full arrays for training (git-ignored)
  data/sample/lgg/*.png        ~200 committed sample slices (256x256 originals + masks) from test patients
  data/sample/lgg/manifest.csv

If the full dataset is not present, run scripts/download_data.py first.
"""
import csv
import os
import random
from pathlib import Path

import numpy as np
from PIL import Image

ROOT = Path(os.getenv("BIOSIGHT_ROOT", Path(__file__).resolve().parents[1]))
RAW = Path(os.getenv("LGG_RAW", ROOT / "data" / "raw" / "lgg" / "kaggle_3m"))
OUT = ROOT / "data" / "raw" / "lgg128.npz"
SAMPLE = ROOT / "data" / "sample" / "lgg"
SEED = 42
SIZE = 128


def main():
    patients = sorted(p for p in RAW.iterdir() if p.is_dir())
    rng = random.Random(SEED)
    order = patients[:]
    rng.shuffle(order)
    n_test = int(round(0.2 * len(order)))
    test_ids = {p.name for p in order[:n_test]}

    xs, ys, pid, split, src = [], [], [], [], []
    for p in patients:
        for img_path in sorted(p.glob("*.tif")):
            if img_path.stem.endswith("_mask"):
                continue
            mask_path = img_path.with_name(img_path.stem + "_mask.tif")
            if not mask_path.exists():
                continue
            img = Image.open(img_path).convert("RGB")
            mask = Image.open(mask_path).convert("L")
            xs.append(np.asarray(img.resize((SIZE, SIZE), Image.BILINEAR), dtype=np.uint8))
            ys.append((np.asarray(mask.resize((SIZE, SIZE), Image.NEAREST)) > 127).astype(np.uint8))
            pid.append(p.name)
            split.append("test" if p.name in test_ids else "train")
            src.append(str(img_path.relative_to(RAW)))

    X, Y = np.stack(xs), np.stack(ys)
    split = np.array(split)
    OUT.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(OUT, X=X, Y=Y, patient=np.array(pid), split=split, src=np.array(src))
    pos = Y.reshape(len(Y), -1).max(1) > 0
    print(f"slices={len(X)} train={int((split == 'train').sum())} test={int((split == 'test').sum())} "
          f"tumour-positive={int(pos.sum())} patients={len(patients)} test_patients={n_test}")

    # committed sample: 140 tumour-positive + 60 negative slices from test patients
    SAMPLE.mkdir(parents=True, exist_ok=True)
    test_idx = np.where(split == "test")[0]
    pos_idx = [i for i in test_idx if pos[i]]
    neg_idx = [i for i in test_idx if not pos[i]]
    rng2 = random.Random(SEED)
    chosen = sorted(rng2.sample(pos_idx, min(140, len(pos_idx))) + rng2.sample(neg_idx, min(60, len(neg_idx))))
    with open(SAMPLE / "manifest.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["file", "mask_file", "patient", "source_slice", "tumour_pixels_256"])
        for i in chosen:
            s = Path(src[i])
            name = s.stem
            img = Image.open(RAW / s).convert("RGB")
            mask = Image.open(RAW / s.with_name(s.stem + "_mask.tif")).convert("L")
            img.save(SAMPLE / f"{name}.png", optimize=True)
            mask.save(SAMPLE / f"{name}_mask.png", optimize=True)
            w.writerow([f"{name}.png", f"{name}_mask.png", pid[i], str(s).replace("\\", "/"),
                        int((np.asarray(mask) > 127).sum())])
    print(f"sample slices written: {len(chosen)} -> {SAMPLE}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/ml/train_mri.py
"""Brain-MRI training: JEPA pre-training (F1), few-label segmentation (F2) and label-efficiency runs (F3).

Stages (results are cached in experiments/mri_metrics.json, so the script can be re-run to resume):
  pretrain-jepa  I-JEPA style latent prediction on unlabelled slices (no masks used)
  pretrain-mae   reconstruction baseline (masked-autoencoder pixel prediction), same encoder and budget
  finetune       ViT + light decoder from {jepa, mae, scratch} and a U-Net baseline, at 10% and 100% of masks

Usage:  python ml/train_mri.py --stage all [--quick]
"""
import argparse
import json
import math
import os
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from timm.models.vision_transformer import Block

sys.path.insert(0, str(Path(__file__).resolve().parent))
from nets import (IMG_GRID, PATCH, MultiScalePredictor, UNet, ViTSeg, block_mask_2d, ema_copy, ema_update,  # noqa: E402
                  image_encoder, sincos_grid)

ROOT = Path(os.getenv("BIOSIGHT_ROOT", Path(__file__).resolve().parents[1]))
DATA = Path(os.getenv("LGG_NPZ", ROOT / "data" / "raw" / "lgg128.npz"))
MODELS = ROOT / "models"
EXP = ROOT / "experiments"
METRICS = EXP / "mri_metrics.json"
SEED = 42
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_metrics():
    return json.loads(METRICS.read_text()) if METRICS.exists() else {"pretrain": {}, "finetune": {}}


def save_metrics(m):
    EXP.mkdir(exist_ok=True)
    METRICS.write_text(json.dumps(m, indent=2))


def load_data():
    d = np.load(DATA)
    X, Y, split = d["X"], d["Y"], d["split"]
    tr, te = np.where(split == "train")[0], np.where(split == "test")[0]
    xf = X[tr].astype(np.float32) / 255.0
    mean = xf.mean(axis=(0, 1, 2)).tolist()
    std = xf.std(axis=(0, 1, 2)).tolist()
    return X, Y, tr, te, {"mean": mean, "std": std}


def to_tensor(xb, norm):
    x = torch.from_numpy(xb).permute(0, 3, 1, 2).float() / 255.0
    m = torch.tensor(norm["mean"]).view(1, 3, 1, 1)
    s = torch.tensor(norm["std"]).view(1, 3, 1, 1)
    return ((x - m) / s).to(DEV)


def cosine_lr(step, total, base, warmup):
    if step < warmup:
        return base * (step + 1) / warmup
    return base * 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))


def batches(idx, bs, rng, drop_last=True):
    p = rng.permutation(idx)
    stop = len(p) - (len(p) % bs if drop_last else 0)
    for i in range(0, stop, bs):
        yield p[i:i + bs]


# ---------------------------------------------------------------- pre-training

def pretrain_jepa(X, tr, norm, epochs, bs):
    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)
    gen = torch.Generator().manual_seed(SEED)
    enc = image_encoder().to(DEV)
    tgt = ema_copy(enc)
    pred = MultiScalePredictor(enc.dim, IMG_GRID, (2, 2)).to(DEV)
    params = list(enc.parameters()) + list(pred.parameters())
    opt = torch.optim.AdamW(params, lr=5e-4, weight_decay=0.04)
    steps_per_epoch = len(tr) // bs
    total = epochs * steps_per_epoch
    step, hist = 0, []
    t0 = time.time()
    for ep in range(epochs):
        losses = []
        for b in batches(tr, bs, rng):
            x = to_tensor(X[b], norm)
            if rng.random() < 0.5:
                x = x.flip(-1)
            ctx_idx, tgt_idx = (i.to(DEV) for i in block_mask_2d(IMG_GRID, gen))
            for g in opt.param_groups:
                g["lr"] = cosine_lr(step, total, 5e-4, steps_per_epoch)
            with torch.no_grad():
                h = tgt(x)
                h = F.layer_norm(h, (h.shape[-1],))[:, tgt_idx]
            z = enc(x, keep_idx=ctx_idx)
            p = pred(z, ctx_idx, tgt_idx)
            loss = F.smooth_l1_loss(p, h)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 3.0)
            opt.step()
            ema_update(tgt, enc, 0.996 + (1.0 - 0.996) * step / total)
            losses.append(loss.item())
            step += 1
        with torch.no_grad():
            feat_std = tgt(to_tensor(X[tr[:64]], norm)).std(0).mean().item()
        hist.append({"epoch": ep + 1, "loss": float(np.mean(losses)), "target_feature_std": feat_std})
        print(f"[jepa] epoch {ep + 1}/{epochs} loss={hist[-1]['loss']:.4f} feat_std={feat_std:.3f} "
              f"({time.time() - t0:.0f}s)", flush=True)
    # the EMA target encoder is the one kept for downstream use (as in I-JEPA)
    torch.save({"encoder": tgt.cpu().state_dict(), "norm": norm}, MODELS / "mri_jepa_encoder.pt")
    return {"epochs": epochs, "history": hist, "seconds": round(time.time() - t0, 1)}


class MAEDecoder(nn.Module):
    def __init__(self, enc_dim, dim=96, depth=2, heads=4):
        super().__init__()
        self.inp = nn.Linear(enc_dim, dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.register_buffer("pos", sincos_grid(IMG_GRID, dim)[None], persistent=False)
        self.blocks = nn.ModuleList([Block(dim, heads, qkv_bias=True) for _ in range(depth)])
        self.out = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, PATCH * PATCH * 3))
        nn.init.normal_(self.mask_token, std=0.02)

    def forward(self, z, vis_idx, n):
        b = z.shape[0]
        t = self.mask_token.expand(b, n, -1).clone()
        t[:, vis_idx] = self.inp(z)
        t = t + self.pos
        for blk in self.blocks:
            t = blk(t)
        return self.out(t)


def patchify(x):
    b, c, h, w = x.shape
    x = x.reshape(b, c, h // PATCH, PATCH, w // PATCH, PATCH).permute(0, 2, 4, 3, 5, 1)
    return x.reshape(b, (h // PATCH) * (w // PATCH), PATCH * PATCH * c)


def pretrain_mae(X, tr, norm, epochs, bs):
    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED + 1)
    enc = image_encoder().to(DEV)
    dec = MAEDecoder(enc.dim).to(DEV)
    params = list(enc.parameters()) + list(dec.parameters())
    opt = torch.optim.AdamW(params, lr=5e-4, weight_decay=0.05)
    n = IMG_GRID[0] * IMG_GRID[1]
    steps_per_epoch = len(tr) // bs
    total = epochs * steps_per_epoch
    step, hist = 0, []
    t0 = time.time()
    for ep in range(epochs):
        losses = []
        for b in batches(tr, bs, rng):
            x = to_tensor(X[b], norm)
            if rng.random() < 0.5:
                x = x.flip(-1)
            perm = torch.from_numpy(rng.permutation(n)).to(DEV)
            vis_idx, mask_idx = perm[: n // 4].sort().values, perm[n // 4:].sort().values
            for g in opt.param_groups:
                g["lr"] = cosine_lr(step, total, 5e-4, steps_per_epoch)
            target = patchify(x)
            target = (target - target.mean(-1, keepdim=True)) / (target.var(-1, keepdim=True) + 1e-6).sqrt()
            out = dec(enc(x, keep_idx=vis_idx), vis_idx, n)
            loss = F.mse_loss(out[:, mask_idx], target[:, mask_idx])
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 3.0)
            opt.step()
            losses.append(loss.item())
            step += 1
        hist.append({"epoch": ep + 1, "loss": float(np.mean(losses))})
        print(f"[mae] epoch {ep + 1}/{epochs} loss={hist[-1]['loss']:.4f} ({time.time() - t0:.0f}s)", flush=True)
    torch.save({"encoder": enc.cpu().state_dict(), "norm": norm}, MODELS / "mri_mae_encoder.pt")
    return {"epochs": epochs, "history": hist, "seconds": round(time.time() - t0, 1)}


# ---------------------------------------------------------------- segmentation

def dice_loss(logits, y):
    p = torch.sigmoid(logits)
    inter = (p * y).sum((1, 2, 3))
    return (1 - (2 * inter + 1) / (p.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1)).mean()


@torch.no_grad()
def evaluate(model, X, Y, idx, norm, bs=64):
    model.eval()
    tp = fp = fn = 0
    per_slice = []
    for i in range(0, len(idx), bs):
        b = idx[i:i + bs]
        pr = (torch.sigmoid(model(to_tensor(X[b], norm))) > 0.5).squeeze(1).cpu().numpy()
        gt = Y[b].astype(bool)
        tp += int((pr & gt).sum())
        fp += int((pr & ~gt).sum())
        fn += int((~pr & gt).sum())
        for p1, g1 in zip(pr, gt):
            if g1.any():
                per_slice.append(2 * (p1 & g1).sum() / (p1.sum() + g1.sum()))
    model.train()
    return {"dice": round(2 * tp / max(1, 2 * tp + fp + fn), 4), "iou": round(tp / max(1, tp + fp + fn), 4),
            "mean_dice_tumour_slices": round(float(np.mean(per_slice)), 4)}


def build_seg(kind):
    if kind == "unet":
        return UNet(), None
    model = ViTSeg(image_encoder())
    if kind in ("jepa", "mae"):
        ck = torch.load(MODELS / f"mri_{kind}_encoder.pt", map_location="cpu")
        model.encoder.load_state_dict(ck["encoder"])
    return model, model.encoder


def finetune(kind, frac, X, Y, tr, te, norm, steps, bs):
    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)
    lab = np.random.default_rng(SEED).permutation(tr)[: max(bs, int(round(frac * len(tr))))]
    model, enc = build_seg(kind)
    model.to(DEV)
    if enc is not None:
        enc_params = list(enc.parameters())
        enc_ids = {id(p) for p in enc_params}
        rest = [p for p in model.parameters() if id(p) not in enc_ids]
        enc_lr = 2e-4 if kind in ("jepa", "mae") else 5e-4
        groups = [{"params": enc_params, "base": enc_lr}, {"params": rest, "base": 1e-3}]
    else:
        groups = [{"params": list(model.parameters()), "base": 1e-3}]
    opt = torch.optim.AdamW([{"params": g["params"], "lr": g["base"]} for g in groups], weight_decay=0.01)
    bases = [g["base"] for g in groups]
    t0 = time.time()
    step = 0
    model.train()
    while step < steps:
        for b in batches(lab, bs, rng):
            if step >= steps:
                break
            x = to_tensor(X[b], norm)
            y = torch.from_numpy(Y[b]).float().unsqueeze(1).to(DEV)
            if rng.random() < 0.5:
                x, y = x.flip(-1), y.flip(-1)
            for g, base in zip(opt.param_groups, bases):
                g["lr"] = cosine_lr(step, steps, base, max(1, steps // 20))
            logits = model(x)
            loss = F.binary_cross_entropy_with_logits(logits, y) + dice_loss(logits, y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            step += 1
            if step % 100 == 0:
                print(f"[{kind}-{int(frac * 100)}%] step {step}/{steps} loss={loss.item():.4f} "
                      f"({time.time() - t0:.0f}s)", flush=True)
    res = evaluate(model, X, Y, te, norm)
    res.update({"labelled_slices": int(len(lab)), "steps": steps, "seconds": round(time.time() - t0, 1)})
    model.cpu()
    torch.save({"model": model.state_dict(), "kind": kind, "frac": frac, "norm": norm, "metrics": res},
               MODELS / f"mri_seg_{kind}_{int(frac * 100)}.pt")
    print(f"[{kind}-{int(frac * 100)}%] test {res}", flush=True)
    return res


def make_plots(m):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    ft = m["finetune"]
    kinds = [k for k in ("jepa", "mae", "scratch", "unet") if f"{k}_10" in ft or f"{k}_100" in ft]
    labels = {"jepa": "ViT + JEPA pre-train", "mae": "ViT + MAE (reconstruction)", "scratch": "ViT from scratch",
              "unet": "U-Net (supervised)"}
    fig, ax = plt.subplots(figsize=(7, 4))
    w = 0.2
    for i, k in enumerate(kinds):
        vals = [ft.get(f"{k}_{p}", {}).get("dice", 0) for p in (10, 100)]
        ax.bar(np.arange(2) + i * w, vals, w, label=labels[k])
    ax.set_xticks(np.arange(2) + w * (len(kinds) - 1) / 2)
    ax.set_xticklabels(["10% of masks", "100% of masks"])
    ax.set_ylabel("Test Dice")
    ax.set_title("Label efficiency - brain MRI tumour segmentation")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(EXP / "mri_label_efficiency.png", dpi=120)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(6, 3.5))
    for k in ("jepa", "mae"):
        if k in m["pretrain"]:
            h = m["pretrain"][k]["history"]
            ax.plot([e["epoch"] for e in h], [e["loss"] for e in h], marker="o", label=k.upper())
    ax.set_xlabel("epoch")
    ax.set_ylabel("pre-training loss")
    ax.legend()
    fig.tight_layout()
    fig.savefig(EXP / "mri_pretrain_loss.png", dpi=120)
    plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", default="all", choices=["all", "pretrain-jepa", "pretrain-mae", "finetune", "plots"])
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--steps-full", type=int, default=500)
    ap.add_argument("--steps-few", type=int, default=300)
    ap.add_argument("--bs", type=int, default=32)
    ap.add_argument("--only", default="", help="comma list of finetune runs, e.g. jepa_10,unet_100")
    ap.add_argument("--force", action="store_true")
    ap.add_argument("--threads", type=int, default=0)
    a = ap.parse_args()
    if a.threads:
        torch.set_num_threads(a.threads)
    MODELS.mkdir(exist_ok=True)
    EXP.mkdir(exist_ok=True)
    X, Y, tr, te, norm = load_data()
    print(f"train slices={len(tr)} test slices={len(te)} threads={torch.get_num_threads()}", flush=True)
    m = load_metrics()
    m["data"] = {"train_slices": int(len(tr)), "test_slices": int(len(te)), "image_size": 128, "patch": PATCH}
    if a.stage in ("all", "pretrain-jepa") and (a.force or "jepa" not in m["pretrain"]):
        m["pretrain"]["jepa"] = pretrain_jepa(X, tr, norm, a.epochs, a.bs)
        save_metrics(m)
    if a.stage in ("all", "pretrain-mae") and (a.force or "mae" not in m["pretrain"]):
        m["pretrain"]["mae"] = pretrain_mae(X, tr, norm, a.epochs, a.bs)
        save_metrics(m)
    if a.stage in ("all", "finetune"):
        runs = [(k, 0.1) for k in ("jepa", "unet", "mae", "scratch")] + [(k, 1.0) for k in ("jepa", "unet", "mae", "scratch")]
        only = set(filter(None, a.only.split(",")))
        for kind, frac in runs:
            key = f"{kind}_{int(frac * 100)}"
            if (only and key not in only) or (key in m["finetune"] and not a.force):
                continue
            steps = a.steps_few if frac < 1 else a.steps_full
            m["finetune"][key] = finetune(kind, frac, X, Y, tr, te, norm, steps, a.bs)
            save_metrics(m)
    make_plots(m)
    save_metrics(m)


if __name__ == "__main__":
    main()


In [ ]:
!python /kaggle/working/ml/prepare_data.py

In [ ]:
# GPU budget: more pre-training epochs and fine-tuning steps than the CPU run
!python /kaggle/working/ml/train_mri.py --stage all --epochs 40 --steps-full 2000 --steps-few 800 --bs 64

In [ ]:
import json; print(json.dumps(json.load(open('/kaggle/working/experiments/mri_metrics.json'))['finetune'], indent=2))
from IPython.display import Image, display
display(Image('/kaggle/working/experiments/mri_label_efficiency.png'))